# Assignment 7 — Transfer Learning with AlexNet, VGG16, ResNet50 and EfficientNetB0

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and fair comparison

Compare four ImageNet-pretrained CNNs on CIFAR-10. Each backbone is
frozen and only its final classifier is trained. All models use the same
subset, input size, optimizer, batch size, and epochs.

PyTorch/torchvision is used because it provides official pretrained
weights for **all four requested architectures**, including AlexNet.
This is feature-extraction transfer learning; optional fine-tuning can
unfreeze the last block after the comparison.


In [ ]:
# Colab already includes PyTorch. Uncomment if torchvision is unavailable.
# %pip install -q -U torch torchvision
import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# Pretrained-weight metadata supplies the correct ImageNet preprocessing.
common_weights = models.ResNet50_Weights.DEFAULT
transform = common_weights.transforms(crop_size=224, resize_size=232)

full_train = datasets.CIFAR10("data", train=True, download=True, transform=transform)
full_test = datasets.CIFAR10("data", train=False, download=True, transform=transform)
rng = np.random.default_rng(SEED)
train_ids = rng.choice(len(full_train), 8000, replace=False)
test_ids = rng.choice(len(full_test), 2000, replace=False)
train_loader = DataLoader(Subset(full_train, train_ids), batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(Subset(full_test, test_ids), batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)
class_names = full_train.classes


In [ ]:
def make_model(name, num_classes=10):
    if name == "AlexNet":
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == "VGG16":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "EfficientNetB0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(name)

    for parameter in model.parameters():
        parameter.requires_grad = False
    head = model.fc if name == "ResNet50" else model.classifier[-1]
    for parameter in head.parameters():
        parameter.requires_grad = True
    return model.to(device)

def train_head(model, epochs=3):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    start = time.perf_counter()
    for epoch in range(epochs):
        # Keep the frozen backbone (including BatchNorm/Dropout) in inference mode.
        model.eval(); correct = total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)
            loss.backward(); optimizer.step()
            correct += (logits.argmax(1) == labels).sum().item(); total += len(labels)
        print(f"epoch {epoch+1}: train accuracy={correct/total:.3f}")
    return time.perf_counter() - start

@torch.no_grad()
def evaluate(model):
    model.eval(); correct = total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += len(labels)
    return correct / total


In [ ]:
rows = []
for name in ["AlexNet", "VGG16", "ResNet50", "EfficientNetB0"]:
    print(); print(name)
    model = make_model(name)
    seconds = train_head(model, epochs=3)
    accuracy = evaluate(model)
    rows.append({
        "model": name,
        "test_accuracy": accuracy,
        "total_parameters_m": sum(p.numel() for p in model.parameters()) / 1e6,
        "trainable_parameters_m": sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6,
        "train_seconds": seconds,
    })
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

comparison = pd.DataFrame(rows).sort_values("test_accuracy", ascending=False)
display(comparison.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison.plot(x="model", y="test_accuracy", kind="bar", legend=False, ax=axes[0])
axes[0].set_ylim(0, 1); axes[0].set_ylabel("Accuracy"); axes[0].set_title("Accuracy")
comparison.plot(x="model", y="train_seconds", kind="bar", legend=False, ax=axes[1], color="orange")
axes[1].set_ylabel("Seconds"); axes[1].set_title("Training time")
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


## Discussion

Accuracy measures predictive quality, while time and parameter count
measure computational cost. VGG16 and AlexNet have large dense heads;
ResNet uses residual connections; EfficientNet balances depth, width,
and resolution. Because only the new head is trained, a second experiment
could unfreeze the last block and use a small learning rate such as `1e-5`.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
